# PDF Resume RAG Practice

This notebook applies a RAG workflow to a PDF resume: load and split the PDF, embed and store its chunks in Chroma, retrieve passages for a question, and answer from that context. It is useful for seeing how a real document moves through ingestion, indexing, retrieval, and generation; inspect retrieved chunks when an answer is incomplete.

### We will load PDF resume first then will procedd to complete rag pipeline


In [1]:
import os
from dotenv import load_dotenv

# Open AI - Model to get response
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/12/zt3xcr4d475gzrfc596w39xm0000gn/T/ipykernel_8414/2807646982.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# First Point - Let's load the pdf file using PyPDFLoader

data_loading = PyPDFLoader("../data/pdf/Ajay-Sharma.pdf")
pdf_data = data_loading.load()
pdf_data[0].page_content


'Ajay\nSharma\nA\nfront-end\ndeveloper\nwith\n5+\nyears\nof\nindustrial\nexperience\nin\ntech\nstartups\nand\nservice\nprovider\ncompanies\nin\nfront-end\ndevelopment.\nGurugram\nHaryana\n9826182195\najysharma704@gmail.com\nEXPERIENCE\nPersistent\nSystem\n,\nGurugram—\nFrontEnd\nDeveloper,\nJAN\n2024\n-\nPRESENT\nWorking\nas\na\nfront-end\ndeveloper.\nAPI\nMigration,\nCode\nMigration,\nCreated\nnew\nfeatures\nfor\nexisting\ntools,\nand\nresponsible\nfor\nwriting\nreadable,\noptimized\ncode\nin\nthe\nfront-end.\nCode\nreviewer,\ndeployment\non\ntest\nand\nstage\nenv.\nResponsibility\n-\nwork\non\nAPI\nMigration,\nCode\nModernisation,\nand\nbuild\nnew\nfeatures\nwith\nunit\ntest\ncoverage\nabove\n85%\nProject\nDomain\n-\nIntuit\n|\nTaxation\nand\nAccounting\nTech\nUsed\n-\nReactJs,\nJavascript,\nStyledComponent,\nGraphQl,\nGit\nXebia\nIt\nArchitect\n,\nGurugram—\nFrontEnd\nDeveloper,\nSEPT\n2021-\nDEC\n2023\nI\nam\nworking\nas\na\nfront-end\ndeveloper.\nAPI\nIntegration\nwith\nthe\nBacke

In [3]:
# Split the data into chunks

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", " ", "."],
    chunk_size=100,
    chunk_overlap=20,
)

chunks = splitter.split_documents(pdf_data)

print(f"Splitted PDF content into {len(chunks)} Chunks")
print(f"First chunk content {chunks[0].page_content}")

Splitted PDF content into 24 Chunks
First chunk content Ajay
Sharma
A
front-end
developer
with
5+
years
of
industrial
experience
in
tech
startups
and
service
provider
companies
in
front-end
development


In [4]:
# Let's create vectors by using embedding
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

hf_huggingface = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)
vectorstore = Chroma.from_documents(
    documents=chunks,
    persist_directory="persistent_db",
    embedding=hf_huggingface,
    collection_name="chromadb-1"
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


/Users/ajaysharma/Desktop/AI/RAGAI/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 19:49:09.888000 8414 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11986.65it/s]


In [5]:
# maintaning history of chat to rewrite query
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Given the conversation history and latest question so please rewrite question as per details! Do not Answer" ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

In [6]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
llm = ChatOpenAI(
    model="gpt-4o-mini"
)
history_aware_chain = create_history_aware_retriever(llm, retriever, rewrite_prompt)

anser_prompt = ChatPromptTemplate(
    [
        ("system", "Answer using the retrieved context, if context not available please Say, You dont have context! \n\n retrieved context:\n{context}"),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}")
    ]
)

document_chain = create_stuff_documents_chain(llm, anser_prompt)

retrieval = create_retrieval_chain(history_aware_chain, document_chain)
retrieval


# create_stuff_documents_chain(
#     retriever,
#     llm=llm,
#     rewrite_prompt
# )

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x174324830>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.

In [7]:
# Create reusable ask function
chat_history=[]
def ask(question: str) -> dict:
    result = retrieval.invoke({"input":question, "chat_history":chat_history})
    chat_history.extend([
                HumanMessage(content=question),
                AIMessage(content=result["answer"]),
            ])
    return result

In [12]:
response = ask("What is Ajay qualifications?")
response

response1 = ask("When passed MCA ? and froom which college ?")
response1


{'input': 'When passed MCA ? and froom which college ?',
 'chat_history': [HumanMessage(content='What is Ajay qualifications?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Ajay has the following qualifications:\n\n1. MCA (Master of Computer Applications) from MNNIT, Allahabad (July 2015 - June 2018) with a grade of 6.7/10.\n2. B.Sc in Computer Science from Jiwaji University, Gwalior (July 2010 - June 2013) with a grade of 62/100.\n3. Completed 12th grade from Excellence Boys H.S. School, Joura (2008 - 2009) with a grade of 58.3/100.\n4. Completed 10th grade from Excellence Boys H.S. School, Joura (2006 - 2007) with a grade of 58.4/100.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is Ajay qualifications?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='You dont have context!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(conte